In [64]:
import pandas as pd
import torch 
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [10]:
live = pd.read_csv('../data/samples/trial/live_metrics.csv')
initial = pd.read_csv('../data/samples/trial/intial_statement.csv')
verbose = pd.read_csv('../data/samples/trial/verbose_statements.csv')

In [11]:
live = live.drop(columns=['Unnamed: 0'])

In [42]:
def normalize(diction : dict):
    diction = (diction)/(diction.max())
    return diction

In [56]:
norm = live.dropna()
norm = norm.iloc[:, 3:]
norm = norm.drop(columns=['Current Time'])
norm = normalize(norm)

In [58]:
np.isnan(norm).any(axis=0)
norm

,GPU Utilization (%),Power Draw (Watts),GPU Temp (°C),GPU Current Clock (MHz),Memory Current Clock (MHz),Memory Allocation Used (MB),Memory Utilization (%),Time Delta,Iteration,GPU Clock Utilization,Memory Clock Utilization
0,0.000000,0.118825,0.488889,0.097826,1.0,0.001415,0.000000,0.001779,0.001923,0.097826,1.0
1,0.000000,0.118825,0.488889,0.097826,1.0,0.001415,0.000000,0.001779,0.003846,0.097826,1.0
2,0.000000,0.118825,0.488889,0.097826,1.0,0.001415,0.000000,0.001779,0.005769,0.097826,1.0
3,0.000000,0.118825,0.488889,0.097826,1.0,0.001415,0.000000,0.001779,0.007692,0.097826,1.0
4,0.000000,0.118825,0.488889,0.097826,1.0,0.001415,0.000000,0.003559,0.009615,0.097826,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2382,0.941176,0.961409,0.977778,1.000000,1.0,1.000000,0.982456,0.998221,0.113462,1.000000,1.0
2383,0.941176,0.961409,0.977778,1.000000,1.0,1.000000,0.982456,0.998221,0.115385,1.000000,1.0
2384,0.941176,0.961409,0.977778,1.000000,1.0,1.000000,0.982456,0.998221,0.117308,1.000000,1.0
2385,0.941176,0.961409,0.977778,1.000000,1.0,1.000000,0.982456,0.998221,0.119231,1.000000,1.0


In [54]:
response_time = verbose['total_duration'].tolist()
response_time = response_time[:-1]
reset_iters = live[live['Iteration'] == 1].index.tolist()

gpu_util_avg = []
memory_util_avg = []
clock_util_avg = []

for i in range(1, len(reset_iters)):
    temp_metrics = live.dropna()
    gpu_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 3].tolist()
    memeory_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], 9].tolist()
    clock_util_prompt= temp_metrics.iloc[reset_iters[i-1]:reset_iters[i], -2].tolist()
    gpu_util_avg.append(np.average(gpu_util_prompt))
    memory_util_avg.append(np.average(memeory_util_prompt))
    clock_util_avg.append(np.average(clock_util_prompt))

In [ ]:
for (x, y, z) in zip(gpu_util_avg, memory_util_avg, clock_util_avg):
    print([x, y, z])

In [63]:
x = norm['GPU Utilization (%)'].values
y = norm['Memory Utilization (%)'].values
z = norm['Memory Clock Utilization'].values

In [ ]:
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection='3d')

In [69]:
import matplotlib.tri as tri

triangular = tri.Triangulation(x, y)
triangular.triangles

array([[   0, 2160,   35],
       [2160,  100,   35],
       [   0,   35,   33],
       [  33,   35,  100],
       [1585,  989, 2160],
       [2160,  306, 1585],
       [ 238,  801, 1842],
       [1842,  362,  238],
       [2160,  989,  866],
       [ 866,  100, 2160],
       [ 866, 1709,  100],
       [ 989, 1585, 2101],
       [ 169, 2101,  172],
       [ 172, 2101, 1585],
       [ 309, 1585,  306],
       [ 309,  172, 1585],
       [ 926,  866,  989],
       [ 989, 2101,  926],
       [ 677,  309,  362],
       [ 172,  309,  677],
       [ 866,  926, 1647],
       [1910, 1051, 1709],
       [1709,  866, 1910],
       [ 866, 1647, 1910],
       [ 736, 1239, 1842],
       [1842,  801,  736],
       [ 238,  362,  309],
       [ 309,  306,  238],
       [ 677, 1842, 1239],
       [ 362, 1842,  677],
       [1239, 1176,  677],
       [ 677, 1176,  172],
       [ 740, 1239,  736],
       [ 740, 1176, 1239],
       [1647,  926, 2101],
       [1647, 2101,  169],
       [ 107,  740, 1052],
 

In [12]:
class BenchMark(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.l1 = nn.Sequential(
            nn.Linear(input_features, 32),
            nn.ReLU()
        )
        self.l2 = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU()
        )
        self.l3 = nn.Linear(16, output_features)

    def foward(self, x):
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        return x